In [2]:
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def crawl_webpage_to_md(url, output_file, content_selector=None):
    try:
        # Giả lập User-Agent để tránh bị block bởi một số server
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
        }

        print(f"Đang tải dữ liệu từ: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # Parse HTML
        soup = BeautifulSoup(response.text, 'html.parser')
        element = soup.select_one(content_selector)

        # # Dọn dẹp bớt các thẻ rác thường gặp nếu không dùng selector cụ thể
        # for element in soup(['script', 'style', 'nav', 'footer', 'noscript', 'iframe']):
        #     element.decompose()

        # # Trích xuất nội dung dựa trên CSS Selector
        # if content_selector:
        #     main_content = soup.select_one(content_selector)
        #     if not main_content:
        #         print(f"Lỗi: Không tìm thấy nội dung với selector '{content_selector}'")
        #         return
        #     html_content = str(main_content)
        # else:
        #     # Nếu không có selector, lấy toàn bộ thẻ body
        #     html_content = str(soup.body) if soup.body else str(soup)

        # Chuyển đổi HTML sang Markdown
        # print("Đang chuyển đổi sang định dạng Markdown...")
        # markdown_text = md(html_content, heading_style="ATX", strip=['img', 'a'])
        # Xóa strip=['img', 'a'] nếu bạn muốn giữ lại hình ảnh và link

        # Ghi ra file .md
         # Bước 1: Xoá thẻ nhiễu trước khi convert
        for noise in element.select(', '.join(_NOISE_TAGS)):
            noise.decompose()

        # Bước 2: HTML → Markdown
        raw_md = md_convert(
            str(element),
            heading_style='ATX',        # dùng # ## ### thay vì gạch dưới
            bullets='-',                # danh sách dùng -
            autolinks=True,             # URL trần → <https://…>
            newline_style='backslash',  # <br> → \ (sạch hơn 2 spaces)
            convert_charrefs=True,      # &amp; → &, &nbsp; → space, v.v.
        )

        # Bước 3: Dọn khoảng trắng thừa
        raw_md = re.sub(r'[ \t]+$', '', raw_md, flags=re.MULTILINE)  # trailing spaces
        raw_md = re.sub(r'\n{3,}', '\n\n', raw_md)                   # tối đa 1 dòng trống
        raw_md = raw_md.strip()

        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(raw_md)

        # with open(output_file, 'w', encoding='utf-8') as f:
        #     f.write(markdown_text)

        print(f"Thành công! Đã lưu file tại: {output_file}")

    except requests.exceptions.RequestException as e:
        print(f"Lỗi kết nối mạng/URL: {e}")
    except Exception as e:
        print(f"Có lỗi hệ thống xảy ra: {e}")


# --- CÁCH SỬ DỤNG ---
def fmain():
    # URL mục tiêu
    target_url = "https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html"

    # CSS selector trỏ tới thẻ div/article chứa nội dung chính
    # Thường là '.post-content', '.entry-content', hoặc 'article'
    # Phải inspect (F12) trên trình duyệt để lấy selector chính xác
    target_selector = ".td-post-content"

    output_filename = "0.Kinh_Tieu_Bo.md"

    crawl_webpage_to_md(
        url=target_url,
        output_file=output_filename,
        content_selector=target_selector
    )
fmain()

Đang tải dữ liệu từ: https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html
Thành công! Đã lưu file tại: 0.Kinh_Tieu_Bo.md


In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def extract_links_to_md(url, content_selector, output_file):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
    }

    try:
        print(f"Đang tải trang: {url}")
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')

        # Tìm vùng chứa nội dung chính
        main_content = soup.select_one(content_selector)
        if not main_content:
            print(f"Lỗi: Không tìm thấy vùng nội dung với selector '{content_selector}'")
            return

        # Lọc và lấy các link hợp lệ
        markdown_links = []
        seen_urls = set()

        for a_tag in main_content.find_all('a', href=True):
            href = a_tag['href']

            # Bỏ qua link nội bộ trang (mỏ neo) hoặc link rỗng
            if href.startswith('#') or href.startswith('javascript:'):
                continue

            # Chuyển đổi link tương đối thành tuyệt đối
            absolute_url = urljoin(url, href)

            # Tránh trùng lặp link
            if absolute_url in seen_urls:
                continue
            seen_urls.add(absolute_url)

            # Lấy tiêu đề hiển thị của link
            title = a_tag.get_text(strip=True)
            if not title:
                title = "Xem chi tiết bài viết"

            # Định dạng theo chuẩn Markdown: - [Tiêu đề](URL)
            markdown_links.append(f"- [{title}]({absolute_url})")

        # Ghi danh sách link vào file .md
        if markdown_links:
            with open(output_file, 'w', encoding='utf-8') as f:
                f.write(f"# Danh sách bài viết trích xuất từ {url}\n\n")
                f.write("\n".join(markdown_links))
            print(f"Thành công! Đã lưu {len(markdown_links)} link vào file: {output_file}")
        else:
            print("Không tìm thấy đường link nào trong vùng nội dung được chọn.")

    except Exception as e:
        print(f"Có lỗi xảy ra: {e}")

# --- THỰC THI ---
def fmain():
    target_url = "https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html"

    # Selector bao quanh khu vực mục lục bài viết
    # Với trang này, bạn có thể dùng '.entry-content' hoặc '.post-content'
    target_selector = ".td-post-content"

    output_filename = "danh_sach_links.md"

    extract_links_to_md(target_url, target_selector, output_filename)
fmain()

Đang tải trang: https://phatphapungdung.com/phap-bao/kinh-tieu-bo-thich-minh-chau-111811.html
Thành công! Đã lưu 139 link vào file: danh_sach_links.md


In [ ]:
from pathlib import Path
import re

folder = Path("../../docs/kinhtieubo/thichminhchau/x")

for file in folder.glob("*.md"):
    old_name = file.name

    # kn-071-phan-1-chuong-i.md
    new_name = re.sub(
        r"^(kn-\d{3}-)",
        r"\1tap-10-",
        old_name
    )

    if new_name != old_name:
        file.rename(file.with_name(new_name))
        print(f"{old_name} -> {new_name}")